# Value Function Approximation — Mountain Car

When the state space is large or continuous, tabular $v$ / $q$ is no longer feasible. We approximate values with a parametric function $\hat v(s,w)$ or $\hat q(s,a,w)$ and update the weight vector $w \in \mathbb{R}^d$ by (semi-)gradient methods.

## Goal
Learn an approximate $q$ (and a good greedy policy) on a **continuous** control task, using linear features (tile coding) and optionally a small neural network.

## Environment
Gymnasium `MountainCar-v0`.  
State $s = (\text{position}, \text{velocity})$ continuous.  
Actions: `0` = push left, `1` = no push, `2` = push right.  
Reward $-1$ per step until the flag is reached (or timeout at 200 steps). Discount $\gamma = 1$ is natural.

## Implemented below
1. Tile coding features $x(s)$ and $x(s,a)$
2. Semi-gradient TD(0) (policy evaluation)
3. Helpers: $\varepsilon$-greedy, evaluate, play
4. Semi-gradient SARSA (linear control)
5. Batched linear Q-learning (course-style "Deep Q")
6. Neural Deep Q-network (PyTorch)
7. Comparison and references


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import time

from IPython.display import clear_output, display

np.random.seed(42)
env = gym.make("MountainCar-v0")


## Reminder: why approximate?

Tabular methods store one entry per state (or state–action). Games and robotics have huge / continuous spaces — we cannot fill a table, and we need **generalization** across similar states.

We replace tables by a differentiable approximator parameterized by $w$:
$$\hat v(s,w) \approx v_\pi(s), \qquad \hat q(s,a,w) \approx q_\pi(s,a).$$

**Semi-gradient TD(0)** (policy evaluation). With target $u_t = R_{t+1} + \gamma \hat v(S_{t+1},w)$:
$$w \leftarrow w + \alpha \bigl[u_t - \hat v(S_t,w)\bigr]\, \nabla_w \hat v(S_t,w).$$
It is *semi*-gradient because the target also depends on $w$, but we do not differentiate through it.

**Linear case.** Features $x(s) \in \mathbb{R}^d$ ("attributs" in the course):
$$\hat v(s,w) = w^\top x(s), \qquad \nabla_w \hat v(s,w) = x(s).$$
Then the update simplifies to $w \leftarrow w + \alpha\,[u_t - w^\top x(S_t)]\, x(S_t)$.


## Tile coding (coarse coding)

Raw features $(1, \text{pos}, \text{vel})$ are too weak for Mountain Car. **Tile coding** covers the continuous $(p,v)$ plane with $m$ overlapping grids (*tilings*), each of resolution $n \times n$.

- Normalize position and velocity to $[0,1]$.
- For each tiling $t$, shift by $\delta = t/m$ and activate one tile $(i,j)$.
- Result: sparse $x(s) \in \{0,1\}^{m n^2}$ with exactly $m$ ones.

Indices are clipped to $\{0,\ldots,n-1\}$ so that states on the boundary (e.g. near the flag) do not overflow.

Typical choices: $m \approx 8$ tilings, $n \approx 8$–$10$, and step size $\alpha \approx 0.1/m$.


In [ ]:
def tile_coding(env, s, n=4, m=6):
    p, v = s

    p_min = env.observation_space.low[0]
    p_max = env.observation_space.high[0]
    v_min = env.observation_space.low[1]
    v_max = env.observation_space.high[1]

    p_hat = (p - p_min) / (p_max - p_min)
    v_hat = (v - v_min) / (v_max - v_min)

    x = np.zeros(m * n * n)
    for t in range(m):
        delta = t / m
        i = int(np.floor(n * p_hat + delta))
        j = int(np.floor(n * v_hat + delta))
        i = min(max(i, 0), n - 1)
        j = min(max(j, 0), n - 1)
        k = t * (n * n) + i * n + j
        x[k] = 1
    return x


## Semi-gradient TD(0) — policy evaluation

Estimate $v_\pi$ for a **fixed** policy $\pi$ (not control). Linear approximator $\hat v(s,w) = w^\top x(s)$:
$$w \leftarrow w + \alpha \bigl[R + \gamma \hat v(S',w) - \hat v(S,w)\bigr]\, x(S),$$
with $\hat v(S',w)=0$ if $S'$ is terminal.

This returns weights $w$ for $v$, not a policy. Useful as a building block; for Mountain Car we mainly care about **control** below.


In [ ]:
def semi_gradient_td(env, pi, alpha, gamma, x=tile_coding, n_episodes=1000):
    s0, _ = env.reset()
    w = np.zeros(len(x(env, s0)))
    for _ in range(n_episodes):
        s, _ = env.reset()
        done = False
        while not done:
            a = pi(s)
            s_prime, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            v = w @ x(env, s)
            v_next = 0.0 if done else w @ x(env, s_prime)
            w += alpha * (r + gamma * v_next - v) * x(env, s)
            s = s_prime
    return w


## Features for $q$ and $\varepsilon$-greedy

For control we need $\hat q(s,a,w)$, so features must depend on the **action**. One standard construction: put $x(s)$ in the block of action $a$ inside a vector of size $d \cdot |\mathcal{A}|$:
$$\hat q(s,a,w) = w^\top x(s,a), \qquad \nabla_w \hat q(s,a,w) = x(s,a).$$

**$\varepsilon$-greedy w.r.t. $\hat q(s,\cdot,w)$.** With probability $\varepsilon$, pick a random action; otherwise
$$a = \arg\max_{a'} \hat q(s,a',w).$$
The policy therefore **depends on the current** $w$ (required for SARSA / Q-learning control).


In [ ]:
def tile_coding_sa(env, s, a, n=4, m=6):
    x_s = tile_coding(env, s, n, m)
    n_actions = env.action_space.n
    x = np.zeros(n_actions * len(x_s))

    start = a * len(x_s)
    x[start:start + len(x_s)] = x_s
    return x


def epsilon_greedy(env, s, w, x, epsilon=0.1):
    if np.random.rand() < epsilon:
        return env.action_space.sample()

    q_values = [w @ x(env, s, a) for a in range(env.action_space.n)]
    return int(np.argmax(q_values))


## Semi-gradient SARSA (on-policy control)

Same idea as tabular SARSA, but we update $w$ instead of a table entry:
$$w \leftarrow w + \alpha \bigl[R + \gamma \hat q(S',A',w) - \hat q(S,A,w)\bigr]\, \nabla \hat q(S,A,w).$$
In the linear case $\nabla \hat q = x(S,A)$. If $S'$ is terminal, the target is just $R$.

Actions $A$ and $A'$ are chosen **$\varepsilon$-greedy** with respect to the current $\hat q(\cdot,\cdot,w)$ (on-policy).

Returns $w$; the implied greedy policy is $\pi(s)=\arg\max_a \hat q(s,a,w)$.


In [ ]:
def SARSA_semi_gradient(env, alpha, gamma, pi=epsilon_greedy, x=tile_coding_sa, n_episodes=1000):
    s0, _ = env.reset()
    w = np.zeros(len(x(env, s0, 0)))

    for _ in range(n_episodes):
        s, _ = env.reset()
        a = pi(env, s, w, x)
        while True:
            s_prime, r, terminated, truncated, _ = env.step(a)
            a_prime = pi(env, s_prime, w, x)
            Q = w @ x(env, s, a)
            Q_next = w @ x(env, s_prime, a_prime)

            if terminated or truncated:
                w += alpha * (r - Q) * x(env, s, a)
                break
            else:
                w += alpha * (r + gamma * Q_next - Q) * x(env, s, a)
                s = s_prime
                a = a_prime
    return w


## Batched linear Q-learning (course "Deep Q" slide)

Off-policy target with a **max** backup (like Q-learning):
$$y \leftarrow R \quad \text{if terminal,} \qquad y \leftarrow R + \gamma \max_{a''} \hat q(S',a'',w) \quad \text{otherwise.}$$

Transitions $(S,A,y)$ are collected in a mini-batch of size $N$. With linear $\hat q$, minimize
$$L(w) = \frac{1}{N}\sum_{j=1}^{N}\bigl(y_j - w^\top x(S_j,A_j)\bigr)^2$$
by gradient descent:
$$w \leftarrow w - \alpha \cdot \frac{2}{N}\sum_{j}(w^\top x_j - y_j)\, x_j.$$

This follows the course sketch with a linear approximator (still tile features, not a neural net yet).


In [ ]:
def Deep_Q_linear(env, alpha, gamma, N, pi=epsilon_greedy, x=tile_coding_sa, n_episodes=1000):
    s0, _ = env.reset()
    w = np.zeros(len(x(env, s0, 0)))

    batch_s, batch_a, batch_y = [], [], []

    for _ in range(n_episodes):
        s, _ = env.reset()
        a = pi(env, s, w, x)

        while True:
            s_prime, r, terminated, truncated, _ = env.step(a)

            if terminated or truncated:
                y = r
            else:
                q_next = [w @ x(env, s_prime, a2) for a2 in range(env.action_space.n)]
                y = r + gamma * np.max(q_next)

            batch_s.append(s)
            batch_a.append(a)
            batch_y.append(y)

            if len(batch_s) == N:
                grad = np.zeros(len(w))
                for sj, aj, yj in zip(batch_s, batch_a, batch_y):
                    qj = w @ x(env, sj, aj)
                    grad += (qj - yj) * x(env, sj, aj)
                w -= alpha * (2 / N) * grad
                batch_s, batch_a, batch_y = [], [], []

            if terminated or truncated:
                break

            s = s_prime
            a = pi(env, s, w, x)

    return w


## Helpers

**`evaluate`.** Success rate (flag reached = `terminated`) and mean episode length of a greedy policy.

**`play`.** Visualize a few greedy episodes with `render_mode="rgb_array"`.

Both take a `choose_action(env, s)` callback so the same helpers work for linear $w$ and for a neural net.


In [ ]:
def evaluate(env, choose_action, num_episodes=100):
    successes = 0
    lengths = []
    for _ in range(num_episodes):
        s, _ = env.reset()
        done = False
        steps = 0
        while not done:
            a = choose_action(env, s)
            s, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            steps += 1
        lengths.append(steps)
        if terminated:
            successes += 1
    return successes / num_episodes, np.mean(lengths)


def play(choose_action, n_episodes=3, delay=0.02):
    viz = gym.make("MountainCar-v0", render_mode="rgb_array")
    for ep in range(n_episodes):
        s, _ = viz.reset()
        done = False
        total_r = 0
        steps = 0
        while not done:
            frame = viz.render()
            clear_output(wait=True)
            plt.figure(figsize=(5, 3))
            plt.imshow(frame)
            plt.title(f"épisode {ep+1}  |  steps={steps}  |  reward={total_r}")
            plt.axis("off")
            display(plt.gcf())
            plt.close()

            a = choose_action(viz, s)
            s, r, terminated, truncated, _ = viz.step(a)
            done = terminated or truncated
            total_r += r
            steps += 1
            time.sleep(delay)

        clear_output(wait=True)
        plt.figure(figsize=(5, 3))
        plt.imshow(viz.render())
        status = "succès" if terminated else "échec (timeout)"
        plt.title(f"épisode {ep+1} terminé ({status})  |  steps={steps}  |  reward={total_r}")
        plt.axis("off")
        display(plt.gcf())
        plt.close()
        time.sleep(0.8)
    viz.close()


## Train semi-gradient SARSA

Step size $\alpha \approx 0.1/m$ with $m=6$ tilings. Mountain Car often needs several thousand episodes before the greedy policy reliably reaches the flag.


In [ ]:
alpha = 0.1 / 6
gamma = 1.0

w_sarsa = SARSA_semi_gradient(env, alpha=alpha, gamma=gamma, n_episodes=5000)

rate, mean_len = evaluate(env, lambda e, s: epsilon_greedy(e, s, w_sarsa, tile_coding_sa, 0.0))
print(f"SARSA — succès={rate:.2%}  |  longueur moyenne={mean_len:.1f}")

play(lambda e, s: epsilon_greedy(e, s, w_sarsa, tile_coding_sa, 0.0))


## Train batched linear Q-learning

Same $\alpha$, $\gamma$, and episode budget; batch size $N=32$. Compare success rate and mean length to SARSA.


In [ ]:
w_dqn = Deep_Q_linear(env, alpha=alpha, gamma=gamma, N=32, n_episodes=5000)

rate, mean_len = evaluate(env, lambda e, s: epsilon_greedy(e, s, w_dqn, tile_coding_sa, 0.0))
print(f"Linear Q — succès={rate:.2%}  |  longueur moyenne={mean_len:.1f}")

play(lambda e, s: epsilon_greedy(e, s, w_dqn, tile_coding_sa, 0.0))


## Neural Deep Q-network

Same off-policy / max target and mini-batch idea, but $\hat q(s,\cdot,w)$ is a small MLP on the raw state $(p,v)$ (no hand-crafted tiles). Parameters $w$ are the network weights; the update is ordinary SGD/Adam on the MSE between $Q(s,a)$ and the TD target $y$.

This is a simplified DQN (no replay buffer shuffle, no target network). Still enough to illustrate non-linear FA from the course.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class QNetwork(nn.Module):
    def __init__(self, n_obs, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, s):
        return self.net(s)


def epsilon_greedy_nn(env, s, net, epsilon=0.1):
    if np.random.rand() < epsilon:
        return env.action_space.sample()
    with torch.no_grad():
        q = net(torch.tensor(s, dtype=torch.float32))
        return int(torch.argmax(q).item())


def deep_q_network(env, alpha=1e-3, gamma=0.99, N=32, epsilon=0.1, n_episodes=1000):
    n_obs = env.observation_space.shape[0]
    n_actions = env.action_space.n

    net = QNetwork(n_obs, n_actions)
    optimizer = optim.Adam(net.parameters(), lr=alpha)
    loss_fn = nn.MSELoss()

    batch_s, batch_a, batch_y = [], [], []

    for _ in range(n_episodes):
        s, _ = env.reset()
        a = epsilon_greedy_nn(env, s, net, epsilon)

        while True:
            s_prime, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated

            if done:
                y = float(r)
            else:
                with torch.no_grad():
                    q_next = net(torch.tensor(s_prime, dtype=torch.float32))
                    y = float(r + gamma * torch.max(q_next).item())

            batch_s.append(s)
            batch_a.append(a)
            batch_y.append(y)

            if len(batch_s) == N:
                s_t = torch.tensor(np.array(batch_s), dtype=torch.float32)
                a_t = torch.tensor(batch_a, dtype=torch.long)
                y_t = torch.tensor(batch_y, dtype=torch.float32)

                q_all = net(s_t)
                q_sa = q_all.gather(1, a_t.unsqueeze(1)).squeeze(1)

                loss = loss_fn(q_sa, y_t)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                batch_s, batch_a, batch_y = [], [], []

            if done:
                break

            s = s_prime
            a = epsilon_greedy_nn(env, s, net, epsilon)

    return net


## Train neural DQN

Adam with a small learning rate; $\gamma=0.99$. Evaluate and play with a greedy policy ($\varepsilon=0$).


In [ ]:
net = deep_q_network(env, alpha=1e-3, gamma=0.99, N=32, n_episodes=5000)
rate, mean_len = evaluate(env, lambda e, s: epsilon_greedy_nn(e, s, net, 0.0))
print(f"DQN (NN) — succès={rate:.2%}  |  longueur moyenne={mean_len:.1f}")
play(lambda e, s: epsilon_greedy_nn(e, s, net, 0.0))


## Conclusion

| Method | Role | On / off policy | Approximator |
|---|---|---|---|
| Semi-gradient TD(0) | **Evaluation** of a fixed $\pi$ | on-policy | linear $x(s)$ |
| Semi-gradient SARSA | **Control** | on-policy ($\varepsilon$-greedy) | linear $x(s,a)$ |
| Batched linear Q | **Control** | off-policy (max target) | linear $x(s,a)$ |
| Neural DQN | **Control** | off-policy (max target) | MLP on raw $s$ |

**Takeaways.**

- With FA we learn $w$, not a table; the policy is read greedily from $\hat q(\cdot,\cdot,w)$.
- Good features matter: tile coding is the course example for Mountain Car; raw linear features are usually too weak.
- SARSA bootstraps with $A'$ actually taken; Q-learning / DQN bootstrap with $\max_{a'} \hat q(S',a')$.
- Non-linear FA (NN) is more flexible but less stable; full DQN adds experience replay and a target network (course discussion).
- Convergence guarantees that held for tabular TD are weaker with FA, especially off-policy + non-linear approximators.


## References

1. S. Airiau. *Apprentissage par renforcement*. M2 IASD, Université Paris Dauphine. Lecture: *Approximation de la fonction de valeurs*. [https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/](https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/)
2. R. S. Sutton and A. G. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. MIT Press, 2018. Chapters 9–10 (on-policy prediction with approximation; control with approximation).
3. V. Mnih et al. *Human-level control through deep reinforcement learning*. Nature, 2015. (DQN)
